In [1]:
import pandas as pd
import ast
from pathlib import Path


PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

### Helper Functions

In [2]:
def clean_list_column(series):
    """
    Convert list/JSON-like values into a consistent
    comma-separated string.
    """

    def clean(value):

        if pd.isna(value):
            return ""

        value = str(value).strip()

        # Try to parse Python-list-like strings
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return ", ".join(
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                )

        except (ValueError, SyntaxError):
            pass

        return value

    return series.apply(clean)

In [3]:
def clean_numeric(series):
    """
    Convert numeric-looking values into numeric dtype.
    Invalid values become NaN.
    """
    return pd.to_numeric(series, errors="coerce")

### Movies

In [4]:
movies = pd.read_csv(
    PROCESSED_DIR / "movie_metadata.csv"
)

movies = movies.copy()

movies["release_year"] = pd.to_datetime(movies["release_year"], errors="coerce")

movies["release_year"] = (movies["release_year"].dt.year.astype("Int64"))


movies["revenue"] = clean_numeric(movies["revenue"])
movies["runtime"] = clean_numeric(movies["runtime"])

for column in [
    "genres",
    "production_companies",
    "cast",
    "director"
]:
    movies[column] = clean_list_column(
        movies[column]
    )

movies.to_csv(PROCESSED_DIR / "movie_metadata_clean.csv",index=False)

### Anime

In [7]:
# =========================
# ANIME CLEANING
# =========================

anime = pd.read_csv(
    PROCESSED_DIR / "anime_metadata.csv"
).copy()

print("Original columns:")
print(anime.columns.tolist())


# -------------------------
# Rename columns
# -------------------------

anime.rename(columns={
    "genre_clean": "genres",
    "ranked": "ranking"
}, inplace=True)


# -------------------------
# Numeric fields
# -------------------------

anime["episodes"] = pd.to_numeric(
    anime["episodes"],
    errors="coerce"
)

anime["popularity"] = pd.to_numeric(
    anime["popularity"],
    errors="coerce"
)

anime["ranking"] = pd.to_numeric(
    anime["ranking"],
    errors="coerce"
)


# -------------------------
# Aired
# -------------------------

# Keep original aired information
anime["aired"] = (
    anime["aired"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Extract the first 4-digit year
# from the aired value.

anime["release_year"] = (
    anime["aired"]
    .str.extract(r"(\d{4})", expand=False)
)

anime["release_year"] = pd.to_numeric(
    anime["release_year"],
    errors="coerce"
).astype("Int64")


# -------------------------
# Text fields
# -------------------------

anime["title"] = (
    anime["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

anime["genres"] = (
    anime["genres"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# -------------------------
# Save
# -------------------------

anime.to_csv(
    PROCESSED_DIR / "anime_metadata_clean.csv",
    index=False
)


# -------------------------
# Validation
# -------------------------

print("\nCleaned columns:")
print(anime.columns.tolist())

print("\nShape:")
print(anime.shape)

print("\nData types:")
print(anime.dtypes)

print("\nMissing values:")
print(anime.isna().sum())

Original columns:
['title', 'genres', 'aired', 'episodes', 'popularity', 'ranking']

Cleaned columns:
['title', 'genres', 'aired', 'episodes', 'popularity', 'ranking', 'release_year']

Shape:
(18336, 7)

Data types:
title               str
genres              str
aired               str
episodes        float64
popularity        int64
ranking         float64
release_year      Int64
dtype: object

Missing values:
title              0
genres             0
aired              0
episodes         535
popularity         0
ranking         2827
release_year     313
dtype: int64


In [8]:
print(anime.columns.tolist())
print(anime["aired"].head(10).to_list())

['title', 'genres', 'aired', 'episodes', 'popularity', 'ranking', 'release_year']
['Oct 4, 2015 to Mar 27, 2016', 'Oct 10, 2014 to Mar 20, 2015', 'Jul 7, 2017 to Sep 29, 2017', 'Apr 5, 2009 to Jul 4, 2010', 'Jan 6, 2017', 'Jan 7, 2019 to Apr 1, 2019', 'Jul 20, 2001', 'Apr 6, 2019 to Sep 28, 2019', 'Aug 12, 2017 to Aug 13, 2017', 'Apr 6, 2008 to Sep 28, 2008']


### Music

In [9]:
# =========================
# MUSIC CLEANING
# =========================

music = pd.read_csv(
    PROCESSED_DIR / "music_metadata.csv"
).copy()

# Rename raw columns to normalized names
music.rename(columns={
    "track_artist": "artist",
    "track_album_name": "album",
    "duration_min": "duration_minutes"
}, inplace=True)

# Check that renaming actually happened
print("Columns after rename:")
print(music.columns.tolist())

# -------------------------
# Numeric fields
# -------------------------

music["duration_minutes"] = pd.to_numeric(
    music["duration_minutes"],
    errors="coerce"
)

music["release_year"] = pd.to_numeric(
    music["release_year"],
    errors="coerce"
).astype("Int64")

# -------------------------
# Date
# -------------------------

music["track_album_release_date"] = pd.to_datetime(
    music["track_album_release_date"],
    errors="coerce"
)

# -------------------------
# Text fields
# -------------------------

for column in [
    "track_name",
    "artist",
    "album",
    "playlist_genre",
    "playlist_subgenre"
]:
    music[column] = (
        music[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# -------------------------
# Save
# -------------------------

music.to_csv(
    PROCESSED_DIR / "music_metadata_clean.csv",
    index=False
)

# -------------------------
# Validation
# -------------------------

print("\nShape:")
print(music.shape)

print("\nData types:")
print(music.dtypes)

print("\nMissing values:")
print(music.isna().sum())

Columns after rename:
['track_name', 'artist', 'album', 'track_album_release_date', 'playlist_genre', 'playlist_subgenre', 'duration_minutes', 'release_year']

Shape:
(32828, 8)

Data types:
track_name                             str
artist                                 str
album                                  str
track_album_release_date    datetime64[us]
playlist_genre                         str
playlist_subgenre                      str
duration_minutes                   float64
release_year                         Int64
dtype: object

Missing values:
track_name                     0
artist                         0
album                          0
track_album_release_date    1886
playlist_genre                 0
playlist_subgenre              0
duration_minutes               0
release_year                1886
dtype: int64


### Books

In [10]:
# =========================
# BOOKS CLEANING
# =========================

books = pd.read_csv(
    PROCESSED_DIR / "book_metadata.csv"
).copy()

print("Original columns:")
print(books.columns.tolist())


# -------------------------
# Rename columns
# -------------------------

books.rename(columns={
    "Title": "title",
    "Author": "author",
    "Rating": "rating",
    "Isbn": "isbn",
    "genres_clean": "genres",
    "Characters": "characters",
    "Pages": "pages",
    "Publisher": "publisher",
    "PublishDate": "publish_date"
}, inplace=True)


# -------------------------
# Numeric fields
# -------------------------

books["rating"] = pd.to_numeric(
    books["rating"],
    errors="coerce"
)


# Pages can contain values such as:
# "300"
# "1 page"
# etc.

books["pages"] = (
    books["pages"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
)

books["pages"] = pd.to_numeric(
    books["pages"],
    errors="coerce"
)


# -------------------------
# ISBN
# -------------------------

# ISBN is an identifier, not a number.
books["isbn"] = (
    books["isbn"]
    .astype("string")
    .str.strip()
)


# -------------------------
# Publish date
# -------------------------

books["publish_date"] = pd.to_datetime(
    books["publish_date"],
    errors="coerce"
)


# Extract publication year
books["release_year"] = (
    books["publish_date"]
    .dt.year
    .astype("Int64")
)


# -------------------------
# Text fields
# -------------------------

for column in [
    "title",
    "author",
    "genres",
    "characters",
    "publisher"
]:
    books[column] = (
        books[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )


# -------------------------
# Save
# -------------------------

books.to_csv(
    PROCESSED_DIR / "book_metadata_clean.csv",
    index=False
)


# -------------------------
# Validation
# -------------------------

print("\nCleaned columns:")
print(books.columns.tolist())

print("\nShape:")
print(books.shape)

print("\nData types:")
print(books.dtypes)

print("\nMissing values:")
print(books.isna().sum())

Original columns:
['title', 'author', 'rating', 'isbn', 'genres', 'characters', 'pages', 'publisher', 'publish_date']

Cleaned columns:
['title', 'author', 'rating', 'isbn', 'genres', 'characters', 'pages', 'publisher', 'publish_date', 'release_year']

Shape:
(11230, 10)

Data types:
title                      str
author                     str
rating                 float64
isbn                    string
genres                     str
characters                 str
pages                  float64
publisher                  str
publish_date    datetime64[us]
release_year             Int64
dtype: object

Missing values:
title               0
author              0
rating              0
isbn                0
genres              0
characters          0
pages             173
publisher           0
publish_date    10601
release_year    10601
dtype: int64
